In [ ]:
import scipy.stats, scipy.optimize

import matplotlib as mpl

* This description is from a previous project. However, while the numeric values differ, the core explanation remains.

* Also, yes, the code is not my finest work. It was meant for a single, time-sensitive project and I did not have the time to touch it up yet.

---
---
---
---
---
---
---
---
---
---

## Code libraries

---

## Iteration steps

- minima slits size 1x2 = 2
    - this lets us deal with hyper-dense regions, such that we don't do 10k slits in the eye

- count how many points are inside
    - if there are at least 800, we are ok
    - if not, we add (800/bin_count/8)x2=800/bin_count/4 width to the slit (the /8 term is trying to accomodate for the exponential growth in density)
    - we repeat this until we have a density >= 800
    - we save this size and use it as starting value for the next iteration on that side and if we are above 800, we go lower and lower until we are below the 800 mark, point at which we just go back (make sure we save the previous step, as, unlike in the case of increasing the width, now we are not sure where to stop, and we are guaranteed to beak the 800 mark, so, just so we don't have to re-compute one step every time, we just save it in a copy kinda list)

- calculate the delta's and move to the next point
    - the steps should be in units of final_previous_bin_width/2, such that we are still doing a 123 method, at half the bin width
    - the first step is given in an analogous fashion by half the width of the starting point's slit's width

- 800 or 1000
    - we could actually use the minima to be 1000, because it would be too computationally expensive to look for the counts after we extract the corner triangles... but how about we try this once at least...

## How do we actually know which points are in a slith?

- use the left/right top&bottom edges intersections: the left-most and right-most ones, get the closest elements to the q positioning and form a list with the points between, from which we subtract the side triangles
    - but how about when we go vertically?
        - worst case scenario... nothing... the vertical part has few points, but the issue is on the lhs of the eye, where we go almost horizontally and on quite large trnasversals, so we'll have like 30-40% of all points
        - so we also restrict it vertically, which again (even though we didn't mention this before, as it was easier to automatically perceive this horizontally), since the shape of the data set's edges is concave, we can worry-free to just take the highest&lowest points and set them as limits

- how do we go about the diagonal cutting though... since those are basically why we do this whole thing
    - since the left-most vertical line goes lower than the right line and because the two exponential edge curves are set in a way that the curve on the left one is on the left of the curve of the right one, we'll always get at most (once higher or equal to the lowest point in te right vertical line) a horizontal slit, or, for the rest of the data set, a positive incline... thus, we can always check if we have horizontal lines (in which case no triangles to cut) and then just set a filter that checks if the points in the rectangle for their q values are between the two parallels (checking this with the parallels as functions, like we did when we initially cut the values out within the edges)

- now, the second step could also take care of the first one, so there is no need for it... however, this later step requires checking each point by equating the two functions... while the other just requires the same overall iteration, but with a simple greater/lower checking, nothing to equate... plus, the first method does this once horizontally and once vertically, then checks the two losts of indices for matches and only keeps those... this is not computationally heavy and saves a lot more time than it adds by having to equate over the points in the rectangle twice (once in each method, basically thrice, since the first method goes twice, but point taken pls)

- small issue... but would take a stupidly useless amount of computational time to solve... we have slits whose sides are parallel... so when we rotate them, we will get something that is parallel for the histogram fitting, but really we should make the edges follow the curve and then make the fitting function account for this change... also this would have to be done at each iteration, which would change the counts inside so would have to change the edges... giga not worth it

## How do we fit the beta_prime?

- well, our slits are inclined, so if we'd just plot the 1d histogram, we'd be missing the whole point of this procedure... we need to rotate the data or rotate the fitting function to accomodate for this weirder histogram, but really that might as well add just as much computational time and we'd miss on actually seeing how the histograms look like when inclined like that <3
- so we use the given system of coordinates (origin at 0,0), and do as in notes to rotate the slit
- we plot the histogram and fit the beta_prime distribution
- we will get the momentum of the maxima, which we use to get the std's too
- we rotate this mean back into place

---

In [ ]:
def slope_finder(starting_point, up_list_q_n, up_list_dt, down_list_q_n, down_list_dt):
    
    
    # up_list_q_n / up_list_dt_n - the upper edge of the (by-hand) delimitated interval
    # down_list_q_n / down_list_dt_n - the lower edge of the (by-hand) delimitated interval
    
    
    sp0 = starting_point[0]; sp1 = starting_point[1]
    
    # The lists of all the (diagonal) distances to the upper and lower edges from the starting_point.
    sqrt_up   = [np.sqrt((up_list_q_n[  i]-sp0)**2 + (up_list_dt[  i]-sp1)**2) for i in range(len(up_list_q_n  ))]
    sqrt_down = [np.sqrt((down_list_q_n[i]-sp0)**2 + (down_list_dt[i]-sp1)**2) for i in range(len(down_list_q_n))]
    
    
    indx_up   = sqrt_up.index(  min(sqrt_up  ))
    indx_down = sqrt_down.index(min(sqrt_down))
    
    # We solve now two issues: we don't want the slope to blow-up if dx is zero or something very small and we also want to make sure if some two points somehow in our edges, by simply the way they were det apart numerically give a very small negative slope, in reality this should've been >= 0, so we make sure that is the case.
    if (-10**-6 > up_list_q_n[indx_up]-down_list_q_n[indx_down]) or (up_list_q_n[indx_up]-down_list_q_n[indx_down] > 10**-6):
        slope_center = (up_list_dt[indx_up]-down_list_dt[indx_down]) / (up_list_q_n[indx_up]-down_list_q_n[indx_down])
    else:
        # We maintain the sign. Very important is how we use it in the rotator_3000 function. A negative slope can only signal a positive incline, but the used math.atan function only wirks in the I & IV quadrants. So, we must make a special case where negative slope is treated as a II quadrant.
        ss1 = up_list_dt[indx_up]-down_list_dt[indx_down]
        
        sss1 = -1
        if ss1 > abs(ss1): sss1 = 1
        
        ss2 = up_list_q_n[indx_up]-down_list_q_n[indx_down]
        sss2 = -1
        if ss2 > abs(ss2): sss2 = 1
        
        sign = sss1*sss2
        slope_center = sign*10**6
    
    
    if up_list_dt[indx_up]-down_list_dt[indx_down] >= 0:
        if up_list_q_n[indx_up]-down_list_q_n[indx_down] >= 0:
            quadrant = 1
        else:
            quadrant = 2
    else:
        if up_list_q_n[indx_up]-down_list_q_n[indx_down] >= 0:
            quadrant = 4
        else:
            quadrant = 3
    
    
    return slope_center, quadrant

---

In [ ]:
def elements_finder(q, dt, starting_point, minima_slit_size, minima_slit_count, maxima_slit_count, previous_slit_size, length_bin_0):
    
    
    # q / dt - the q / dt positions of the elements in the set (the cut_cut_n one)
    # previous_slit_elements - the result of this function from the previous iteration
    #                        - helps us with looking for the closest element in the list to the guess just made (the starting point)
    # starting_pint - the guess made about where the fit in this region we are determining with this function (and setting its bounds from) is at
    # minima_slit_size - the min/max physical sizes of the slit, as taken in its length (not just along some axis)
    #                  - the minima is good for not staying too much around hyper-dense regions and just to make sure we also get a broader spectrum of data points even where our statistical needs are met (so we smooth-out possible physical deffects in the data and so on)
    # minima_slit_count / maxima_slit_count - the min/max element count that the list we produce with this function must have
    #                                       - the minima is good for maintaining a statistical necessity
    #                                       - the maxima is just a bound so we don't take too much data if we have ALREADY satisfied the minima_slit_size condition... that is, this later condition must always be met... while this maxima_slit_count one is optional, if the minma width slit contains already more data points than it
    # previous_slit_size - it is a good starting point for how wide this iteration's slit should be... so we don't have to start from a too small or too large guess width
    #                    - now, this is the width between the slits' edges... not in the q direction (as it'd be indinite in the vertical section since the edges are parallel)
    #                    - this raises the issue of the inclination when picking this new iteration, izi: it shouldn't be that of the derivative, since it can be that the derivative sends us in a weird direction one step and it just wouldn't make much sense to follow it if we have the edges of the q and dt (_n), whose average is a decent approximation to the path (as we saw), so let us follow the tangent to the slope_center
    
    
    
    
    slit_count = 0
    
    # for the first step
    if previous_slit_size == 0: slit_size = minima_slit_size
    else:                       slit_size = previous_slit_size
    
    avg_slit_count = (minima_slit_count + maxima_slit_count)/2
    
    
    
    ### We look for the closest element to the guess in the actual data set.
    ### Since we advance in the 123 method with separations smaller than the bin size, we are basically guaranteed to find the clsoeset point inside the previous bin.
    
    # for the first step
    lu1 = []
    for i in range(len(q)):
        lu1.append(np.sqrt((q[i]-starting_point[0])**2 + (dt[i]-starting_point[1])**2))
    indx2 = lu1.index(min(lu1))
    closest_number = [q[indx2], dt[indx2]]
    
    
    
    slit_elements = [[closest_number[0]], [closest_number[1]]]
    indx_listo = [-1]
    pm = [1, -1]
    
    i = 0
    
    lq = len(q)-1
    oo  = [True,True]
    oo1 = [True,True]
    # as long as we didn't end counting in this slit
    while True:
        
        # alternating left & right
        for kk in range(2):
            if oo1[kk] and oo[kk]:
                
                if q[indx2]-slit_size/2 <= q[indx2-pm[kk]*i] <= q[indx2]+slit_size/2:
                    if dt[indx2]-length_bin_0 <= dt[indx2-pm[kk]*i] <= dt[indx2]+length_bin_0:
                        slit_elements[0].append(q[indx2-pm[kk]*i])
                        slit_elements[1].append(dt[indx2-pm[kk]*i])
                        indx_listo.append(i)
                else:
                    oo1[kk] = False

                if 0 <= indx2-pm[kk]*(i+1) < lq:
                    oo[kk] = True
                else:
                    oo[kk] = False
        
        i += 1
        
        if ((not oo[0]) or (not oo1[0])) and ((not oo[1]) or (not oo1[1])): break
        
            
            

    
    termk = 0
    death_1 = True
    # as long as we ain't satisfying the conditions at the end of each slit counting trial
    while death_1:
        
        # Just so we don't spend too much time... This is the number of max iterations we want to go over before we give up (in case we don't ind a satisfying slit by then)
        termk += 1
        if termk == 25: break
        
        this_slit_count = len(slit_elements[0])
        
        if minima_slit_count <= this_slit_count <= maxima_slit_count:
            death_1 = False
            break
        
        
        
        elif this_slit_count < minima_slit_count:
            slit_size1 = slit_size * (avg_slit_count/this_slit_count)
            
            # size check
            if slit_size1 < minima_slit_size: slit_size = minima_slit_size
            else:                             slit_size = slit_size1
            
           
            # We don't reset the i.
            oo1 = [True,True]
            while True:
                
                # alternating left & right
                for kk in range(2):
                    if oo1[kk] and oo[kk]:

                        if q[indx2]-slit_size/2 <= q[indx2-pm[kk]*i] <= q[indx2]+slit_size/2:
                            if dt[indx2]-length_bin_0 <= dt[indx2-pm[kk]*i] <= dt[indx2]+length_bin_0:
                                slit_elements[0].append(q[indx2-pm[kk]*i])
                                slit_elements[1].append(dt[indx2-pm[kk]*i])
                                indx_listo.append(i)
                        else:
                            oo1[kk] = False

                        if 0 <= indx2-pm[kk]*(i+1) < lq:
                            oo[kk] = True
                        else:
                            oo[kk] = False
                
                i += 1
                
                if ((not oo[0]) or (not oo1[0])) and ((not oo[1]) or (not oo1[1])): break
            
        
        else:
            slit_size1 = slit_size * (avg_slit_count/this_slit_count)
            
            # size check
            if slit_size1 < minima_slit_size: slit_size = minima_slit_size
            else:                             slit_size = slit_size1
            
            
            indx_listo1 = []
            slit_elements1 = [[], []]
            for j in range(len(slit_elements[0])):
                
                if q[indx2]-slit_size/2 <= slit_elements[0][j] <= q[indx2]+slit_size/2:
                    # No need for dt check... we're looking in the previous list, not adding new elements.
                    indx_listo1.append(indx_listo[j])
                    slit_elements1[0].append(slit_elements[0][j])
                    slit_elements1[1].append(slit_elements[1][j])
                    
            slit_elements = slit_elements1.copy()
            indx_listo = indx_listo1.copy()
            i = max(indx_listo)
            
            
       
    strindx = np.argsort(slit_elements[0])
    slit_elements_fin = [[], []]
    for i in range(len(strindx)):
        slit_elements_fin[0].append(slit_elements[0][strindx[i]])
        slit_elements_fin[1].append(slit_elements[1][strindx[i]])
    
    
    
    return slit_elements_fin, slit_size

---

In [ ]:
def diff_finder(slope_center, quadrant):
    
    # Obviously, if we have a negative tnagent to the slope, it means we're going up when moving to the right. However, we could also go up to the left and just vary the angle enough to be in the forth quadrant.
    
    if quadrant in [1,3]: diff_angle = np.pi/2 - math.atan(slope_center)
    if quadrant in [2,4]: diff_angle = np.pi/2 + math.atan(slope_center)
    
    
    return diff_angle

In [ ]:
def rotator_3000(slit_elements, diff_angle, quadrant):
    
    
    
    slit_elements_rotated = [[], []]
    
    
    
    for i in range(len(slit_elements[0])):
        
        y = slit_elements[1][i]
        x = slit_elements[0][i]
        if x > 10**-6 or x < -10**-6: d =  y/x
        elif -10**-6 <= x < 0:        d = -10**6
        else:                         d =  10**6
        
        
        radius = np.sqrt(x**2 + y**2)
        
        if   x >= 0 and y >= 0: angle_radians =  math.atan(d)
        elif x <  0 and y >= 0: angle_radians =  np.pi + math.atan(d)
        elif x <  0 and y <  0: angle_radians = -np.pi + math.atan(d)
        elif x >= 0 and y <  0: angle_radians =  math.atan(d)
        
        
        if quadrant in [1,3]: pmq =  1
        else:                 pmq = -1
        
        slit_elements_rotated[0].append(radius*np.cos(pmq*diff_angle + angle_radians))
        slit_elements_rotated[1].append(radius*np.sin(pmq*diff_angle + angle_radians))
        
    
    return slit_elements_rotated

---

In [ ]:
def angling_fit(x, *args):
    
    a, b, c = args
    
    ret = ((x-a)/b)**2 + c
    
    return ret

In [ ]:
def duo_norm1(x, *args):
    
    a, mu, sgm, c = args
    
    ret = a * np.e**(-0.5*((x-mu)/sgm)**2) + c
    
    return ret

In [ ]:
def best_g(slit_elements_rotated, ttt):
    
    
    mm = min(slit_elements_rotated[1])
    stepo = abs(max(slit_elements_rotated[1])-mm)/ttt
    
    stepo_bins_x = [mm+stepo/2]
    stepo_bins_y = [0]
    
    i = 1
    while stepo_bins_x[-1] < mm+stepo*(ttt-0.5):
        stepo_bins_x.append(mm+stepo/2+stepo/4*i)
        stepo_bins_y.append(0)
        i += 1
    
    
    ttt1 = len(stepo_bins_y)+2
    for i in range(len(slit_elements_rotated[1])):
        ko = int((slit_elements_rotated[1][i]-mm)//(stepo/4))
        
        if ko == 0:
            stepo_bins_y[   0] += 1
        elif ko == 1:
            stepo_bins_y[   0] += 1
            stepo_bins_y[   1] += 1
        elif ko == 2:
            stepo_bins_y[   0] += 1
            stepo_bins_y[   1] += 1
            stepo_bins_y[   2] += 1
        elif ko == ttt1+1:
            stepo_bins_y[  -1] += 1
        elif ko == ttt1-0:
            stepo_bins_y[  -1] += 1
        elif ko == ttt1-1:
            stepo_bins_y[  -1] += 1
            stepo_bins_y[  -2] += 1
        elif ko == ttt1-2:
            stepo_bins_y[  -1] += 1
            stepo_bins_y[  -2] += 1
            stepo_bins_y[  -3] += 1
        else:
            stepo_bins_y[ko-3] += 1
            stepo_bins_y[ko-2] += 1
            stepo_bins_y[ko-1] += 1
            stepo_bins_y[ko-0] += 1
    
    return stepo_bins_x, stepo_bins_y

In [ ]:
def mean_by_fitting2(slit_elements_rotated, cnc):
    
    
        
    xbins2, ybins1 = best_g(slit_elements_rotated, 25)
    
    
    a   = max(ybins1)
    mu  = xbins2[np.argmax(ybins1)]
    sgm = np.sqrt(sum((x-mu)**2 for x in xbins2)/len(xbins2))
    
    a1   = max(ybins1)
    mu1  = xbins2[np.argmax(ybins1)]+cnc
    sgm1 = np.sqrt(sum((x-mu1)**2 for x in xbins2)/len(xbins2))
    
    
    params = [a, mu, sgm, 0.1]
    bds = ([max(ybins1)*0.1,
            min(xbins2)+(max(xbins2)-min(xbins2))*0.1,
            (max(xbins2)-min(xbins2))/20,
            0],
           [max(ybins1)*2,
            max(xbins2)-(max(xbins2)-min(xbins2))*0.1,
            (max(xbins2)-min(xbins2))/2,
            20])
    
    try:
        fitted_para, s = scipy.optimize.curve_fit(duo_norm1, xbins2, ybins1, p0=params, bounds=bds, maxfev=1000000)
        x = np.linspace(min(xbins2)-5, max(xbins2)+5, 10**5)
    except ValueError:
        fitted_para, s = scipy.optimize.curve_fit(duo_norm1, xbins2, ybins1, p0=params,             maxfev=1000000)
        x = np.linspace(min(xbins2),   max(xbins2),   10**5)

    
    fitted_para1 = fitted_para[:4]
    y1 = duo_norm1(x, *fitted_para1)
    
    slit_max_rotated1 = x[np.argmax(y1)]
    
    sum_y1 = sum(y1)
    slit_max_rotated = (slit_max_rotated1*sum_y1)/sum_y1
    
    slr = np.mean(slit_elements_rotated[0])
    
    
    return slit_max_rotated, slr, fitted_para, 1

In [ ]:
def std_finder(slit_elements_rotated, slit_max_rotated):
    
    std_q = np.sqrt(1/len(slit_elements_rotated[0])*sum([(i-slit_max_rotated)**2 for i in slit_elements_rotated[0]]))
    
    return std_q

---

In [ ]:
def get_quadrant(angle):
    # Normalize the angle to the range [0, 2*pi)
    angle = angle % (2 * math.pi)
    
    # Determine the quadrant
    if 0 <= angle < math.pi / 2:
        return 1
    elif math.pi / 2 <= angle < math.pi:
        return 2
    elif math.pi <= angle < 3 * math.pi / 2:
        return 3
    else:
        return 4

---
---
---